# Minimal Working Baseline (HF Model)
Uses `chbh7051/driver-drowsiness-detection` on local parquet datasets in `data/raw/...` and reports per-dataset + overall metrics.

In [ ]:
from pathlib import Path
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset
from transformers import AutoModelForImageClassification

# ---------------------------
# CONFIG
# ---------------------------
MODEL_ID = "chbh7051/driver-drowsiness-detection"
ROOTS = [
    "data/raw/n7i5x9__driver-drowsiness-dataset",
    "data/raw/akahana__Driver-Drowsiness-Dataset",
]
BATCH_SIZE = 16
NUM_WORKERS = 2
LIMIT_PER_SPLIT = 0  # 0 = full split

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
print("HF_TOKEN in kernel:", bool(os.getenv("HF_TOKEN")))

# ---------------------------
# HELPERS
# ---------------------------
def normalize_label_name(name: str):
    s = str(name).lower().replace("_", " " ).replace("-", " " ).strip()
    if "non drowsy" in s or "not drowsy" in s:
        return 0
    if "drowsy" in s or "sleep" in s or "fatigue" in s or "yawn" in s:
        return 1
    if "alert" in s or "awake" in s or "open" in s or "normal" in s:
        return 0
    return None

def discover_data_files(root: Path):
    files = {}
    for p in sorted((root / "data").glob("*.parquet")):
        split = p.name.split("-")[0]
        files.setdefault(split, []).append(str(p))
    return files

class HFDrowsinessTestDataset(Dataset):
    def __init__(self, split_ds):
        self.ds = split_ds
        names = split_ds.features["label"].names
        self.label_map = {i: normalize_label_name(n) for i, n in enumerate(names)}
        self.valid_idx = [
            i for i in range(len(split_ds))
            if self.label_map.get(int(split_ds[i]["label"])) is not None
        ]

    def __len__(self):
        return len(self.valid_idx)

    def __getitem__(self, idx):
        row = self.ds[self.valid_idx[idx]]
        image = row["image"].convert("RGB")
        y = self.label_map[int(row["label"])]
        return image, int(y)

# Manual preprocessor fallback for legacy checkpoint metadata
IMG_SIZE = 224
manual_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

def processor(images, return_tensors="pt"):
    if not isinstance(images, list):
        images = [images]
    pixel_values = torch.stack([manual_tf(img.convert("RGB")) for img in images], dim=0)
    return {"pixel_values": pixel_values}

def collate_with_processor():
    def _fn(batch):
        images, labels = zip(*batch)
        inputs = processor(list(images), return_tensors="pt")
        y = torch.tensor(labels, dtype=torch.long)
        return inputs, y
    return _fn

def evaluate_loader(model, loader, pred_map, device):
    cm = torch.zeros((2, 2), dtype=torch.long)
    total = 0
    correct = 0

    with torch.no_grad():
        for inputs, y_true in loader:
            inputs = {k: v.to(device) for k, v in inputs.items()}
            y_true = y_true.to(device)

            logits = model(**inputs).logits
            pred_ids = torch.argmax(logits, dim=1)

            mapped_preds = []
            for p in pred_ids.tolist():
                if p in pred_map:
                    mapped_preds.append(pred_map[p])
                elif p in (0, 1):
                    mapped_preds.append(p)
                else:
                    mapped_preds.append(-1)

            for t, p in zip(y_true.tolist(), mapped_preds):
                if p not in (0, 1):
                    continue
                total += 1
                correct += int(p == t)
                cm[t, p] += 1

    acc = correct / total if total > 0 else 0.0
    return acc, cm, total

def print_metrics(name, acc, cm, total):
    tn, fp = int(cm[0, 0]), int(cm[0, 1])
    fn, tp = int(cm[1, 0]), int(cm[1, 1])
    print(f"\n=== {name} ===")
    print(f"samples={total}")
    print(f"accuracy={acc:.4f}")
    print("confusion_matrix (rows=true, cols=pred) [alert, drowsy]:")
    print(cm.numpy())
    print(f"TN={tn} FP={fp} FN={fn} TP={tp}")

# ---------------------------
# LOAD MODEL
# ---------------------------
model = AutoModelForImageClassification.from_pretrained(MODEL_ID, token=os.getenv("HF_TOKEN")).to(DEVICE)
model.eval()

pred_map = {}
for k, v in model.config.id2label.items():
    idx = int(k) if isinstance(k, str) and k.isdigit() else int(k)
    mapped = normalize_label_name(v)
    if mapped is not None:
        pred_map[idx] = mapped

print("Model id2label:", model.config.id2label)
print("Pred map:", pred_map)

# ---------------------------
# EVALUATE EACH DATASET TEST SPLIT
# ---------------------------
overall_cm = torch.zeros((2, 2), dtype=torch.long)
overall_total = 0
overall_correct = 0

for root_str in ROOTS:
    root = Path(root_str)
    data_files = discover_data_files(root)

    if "test" not in data_files:
        print(f"Skipping {root.name}: no test split found")
        continue

    ds = load_dataset("parquet", data_files={"test": data_files["test"]})
    split_ds = ds["test"]

    if LIMIT_PER_SPLIT > 0:
        split_ds = split_ds.select(range(min(LIMIT_PER_SPLIT, len(split_ds))))

    eval_ds = HFDrowsinessTestDataset(split_ds)
    loader = DataLoader(
        eval_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collate_with_processor(),
    )

    acc, cm, total = evaluate_loader(model, loader, pred_map, DEVICE)
    print_metrics(root.name, acc, cm, total)

    overall_cm += cm
    overall_total += total
    overall_correct += int(cm.trace().item())

if overall_total > 0:
    overall_acc = overall_correct / overall_total
    print_metrics("OVERALL", overall_acc, overall_cm, overall_total)
else:
    print("No samples evaluated. Check ROOTS path and dataset files.")
